# 第1章　テンソル（Tensor）の基礎

テンソルは PyTorch の主役データ構造です。ざっくり **「NumPy 配列 ＋ GPU で動く ＋ 自動微分できる」** もの。
スカラー(0次元)・ベクトル(1次元)・行列(2次元)・それ以上(N次元)を統一的に扱います。

この章のゴール：テンソルを**作る・形を見る・計算する・形を変える・GPUに送る**ができる。

> **このノートの使い方**
> - 上から順にセルを実行（Colab/Jupyter ともに `Shift + Enter`）。
> - コードは**少し書き換えて壊して直す**のが一番伸びます。各章末に演習があります。
> - GPU は不要な章が多いです。重い章（CNN）では使い方を案内します。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 1-1. テンソルを作る

代表的な作り方を見ます。`torch.tensor` は既存の数値から、`zeros/ones/randn/arange` は規則的に生成します。

In [ ]:
import torch

a = torch.tensor([1.0, 2.0, 3.0])        # リストから
b = torch.zeros(2, 3)                     # 2x3 のゼロ行列
c = torch.ones(2, 3)                      # 2x3 の1行列
d = torch.randn(2, 3)                     # 標準正規分布の乱数 2x3
e = torch.arange(0, 10, 2)                # 0,2,4,6,8

print("a =", a)
print("b =\n", b)
print("d =\n", d)
print("e =", e)

## 1-2. テンソルの属性：`shape` / `dtype` / `device`

- `shape`（形）… 各次元の大きさ。**最重要**。エラーの9割はここの食い違い。
- `dtype`（型）… `float32` が標準。整数は `int64`(long)。
- `device`（場所）… `cpu` か `cuda`（GPU）。

In [ ]:
x = torch.randn(3, 4)
print("shape :", x.shape)     # torch.Size([3, 4])
print("ndim  :", x.ndim)      # 次元数 = 2
print("dtype :", x.dtype)     # torch.float32
print("device:", x.device)    # cpu

# 型を変える
xi = x.to(torch.int64)
print("int  :", xi.dtype)

## 1-3. 演算：要素ごと・行列積・ブロードキャスト

- `+ - * /` は**要素ごと**（同じ位置同士）。
- 行列積は `@`（または `torch.matmul`）。**要素ごとの掛け算 `*` とは別物**。
- 形が違っても自動で揃えてくれる仕組みが**ブロードキャスト**。

In [ ]:
m = torch.tensor([[1., 2.],
                  [3., 4.]])
n = torch.tensor([[10., 20.],
                  [30., 40.]])

print("要素ごとの積 m*n =\n", m * n)        # 位置ごとに掛ける
print("行列積   m@n =\n", m @ n)            # 行×列

# ブロードキャスト：行ベクトルが各行に自動で足される
row = torch.tensor([100., 200.])
print("broadcast m+row =\n", m + row)

### よく使う集計
`sum / mean / max` などは `dim`（どの軸でまとめるか）が肝。`dim=0` は列方向（行をつぶす）、`dim=1` は行方向（列をつぶす）。

In [ ]:
t = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
print("全体の合計 :", t.sum())
print("列ごと(dim=0):", t.sum(dim=0))   # [5, 7, 9]
print("行ごと(dim=1):", t.sum(dim=1))   # [6, 15]
print("行ごとの平均 :", t.mean(dim=1))

## 1-4. 形を変える：`reshape` / `view` / `squeeze` / `unsqueeze`

- `reshape(...)`（または `view`）… 要素数を保ったまま形を変える。`-1` は「残りを自動計算」。
- `unsqueeze(d)` … 大きさ1の次元を**追加**（バッチ次元を足すときに多用）。
- `squeeze()` … 大きさ1の次元を**削除**。

In [ ]:
x = torch.arange(12)          # 1次元 [0..11]
print("元:", x.shape)
y = x.reshape(3, 4)           # 3x4
print("reshape:", y.shape)
z = x.reshape(2, -1)          # 2x6 （-1 は自動で6）
print("auto -1:", z.shape)

img = torch.randn(28, 28)     # 1枚の画像（28x28）
batch = img.unsqueeze(0)      # 先頭に次元追加 -> (1, 28, 28)
print("unsqueeze:", batch.shape)
print("squeeze  :", batch.squeeze().shape)

## 1-5. インデックスとスライス（NumPy と同じ感覚）

In [ ]:
x = torch.arange(12).reshape(3, 4)
print(x)
print("1行目      :", x[0])        # 最初の行
print("2列目      :", x[:, 1])     # 全行の2列目
print("部分    :\n", x[0:2, 1:3])  # 0..1行, 1..2列
print("条件で抽出 :", x[x > 5])    # 5より大きい要素だけ

## 1-6. NumPy との橋渡し
PyTorch と NumPy は相互変換できます（CPU上ではメモリ共有なので片方を変えると両方変わる点に注意）。

In [ ]:
import numpy as np
np_arr = np.array([1., 2., 3.])
t = torch.from_numpy(np_arr)     # numpy -> tensor
back = t.numpy()                 # tensor -> numpy
print(type(t), t)
print(type(back), back)

## 1-7. GPU に送る（定番の書き方）
「使える環境では GPU、無ければ CPU」を自動で選ぶ書き方を覚えると、どこでも動くコードになります。
**モデルとデータは同じ device に置く**のが鉄則（違うとエラー）。

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("使うデバイス:", device)

x = torch.randn(2, 3).to(device)   # データを device へ
print(x.device)

## 演習 1
1. `torch.arange(24)` から `(2, 3, 4)` のテンソルを作り、`shape` を確認しよう。
2. `(3,3)` の乱数行列を作り、その**行列積** `A @ A` と**要素ごとの積** `A * A` の違いを観察しよう。
3. `(5, 1)` のテンソルと `(1, 4)` のテンソルを足すとどんな形になる？（ブロードキャスト）

In [ ]:
# ここに自分のコードを書いて実行してみよう
